In [2]:
import duckdb
import pandas as pd

In [6]:
sheets=pd.read_excel(r"C:\Users\dines\OneDrive\Desktop\CafeChain_dataset.xlsx", sheet_name=None)

In [8]:
len(sheets)

5

In [13]:
con = duckdb.connect("cafe_chain.duckdb")

In [15]:
for name, data in sheets.items():
    con.execute(f'create or replace table "raw_{name}" as select * from data')

In [17]:
con.sql("show tables")

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ raw_README        │
│ raw_raw_employees │
│ raw_raw_orders    │
│ raw_raw_products  │
│ raw_raw_stores    │
└───────────────────┘

In [30]:
con.sql("select * from raw_raw_orders limit 10").df()

,order_id,store_id,product_id,employee_id,quantity,unit_price,order_date
0,1,1,101,1001,2.0,120,2024-01-03
1,2,2,102,1003,1.0,150,2024-01-05
2,3,3,103,1004,3.0,180,2024-01-07
3,4,1,104,1002,NaN,200,2024-01-09
4,5,2,105,1003,2.0,120,2024-01-11
5,6,3,101,1004,1.0,150,2024-01-14
6,7,99,102,1001,2.0,180,2024-01-16
7,8,1,103,1001,3.0,200,2024-01-18
8,9,2,104,1003,1.0,120,2024-01-21
9,10,3,105,1004,2.0,-50,2024-01-23


In [23]:
con.sql("describe raw_raw_orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,store_id,BIGINT,YES,None,None,None
2,product_id,BIGINT,YES,None,None,None
3,employee_id,BIGINT,YES,None,None,None
4,quantity,DOUBLE,YES,None,None,None
5,unit_price,BIGINT,YES,None,None,None
6,order_date,TIMESTAMP,YES,None,None,None
7,Unnamed: 7,DOUBLE,YES,None,None,None
8,Note: yellow cells = planted issues. Remove th...,DOUBLE,YES,None,None,None


In [31]:
con.sql("select * from raw_raw_orders limit 10").df()

,order_id,store_id,product_id,employee_id,quantity,unit_price,order_date
0,1,1,101,1001,2.0,120,2024-01-03
1,2,2,102,1003,1.0,150,2024-01-05
2,3,3,103,1004,3.0,180,2024-01-07
3,4,1,104,1002,NaN,200,2024-01-09
4,5,2,105,1003,2.0,120,2024-01-11
5,6,3,101,1004,1.0,150,2024-01-14
6,7,99,102,1001,2.0,180,2024-01-16
7,8,1,103,1001,3.0,200,2024-01-18
8,9,2,104,1003,1.0,120,2024-01-21
9,10,3,105,1004,2.0,-50,2024-01-23


In [33]:
# 1. Drop the README table (we don't need it in our database)
con.execute("DROP TABLE raw_README")

# 2. Rename the double-raw tables to single-raw
con.execute("ALTER TABLE raw_raw_orders RENAME TO raw_orders")
con.execute("ALTER TABLE raw_raw_stores RENAME TO raw_stores")
con.execute("ALTER TABLE raw_raw_products RENAME TO raw_products")
con.execute("ALTER TABLE raw_raw_employees RENAME TO raw_employees")

# 3. Check it worked
con.sql("SHOW TABLES").df()

CatalogException: Catalog Error: Table with name raw_README does not exist!
Did you mean "raw_orders"?

In [35]:
con.sql("SHOW TABLES").df()

,name
0,raw_employees
1,raw_orders
2,raw_products
3,raw_stores


In [37]:
con.sql("""select * from raw_orders where quantity<=0""").df()

,order_id,store_id,product_id,employee_id,quantity,unit_price,order_date
0,1,1,101,1001,2.0,120,2024-01-03
1,2,2,102,1003,1.0,150,2024-01-05
2,3,3,103,1004,3.0,180,2024-01-07
3,5,2,105,1003,2.0,120,2024-01-11
4,6,3,101,1004,1.0,150,2024-01-14
5,7,99,102,1001,2.0,180,2024-01-16
6,8,1,103,1001,3.0,200,2024-01-18
7,9,2,104,1003,1.0,120,2024-01-21
8,10,3,105,1004,2.0,-50,2024-01-23
9,11,1,101,1002,2.0,150,2024-01-26


In [44]:
query_quarantine = """
CREATE OR REPLACE TABLE quarantine_orders AS
SELECT 
    *,
    CASE 
        WHEN quantity IS NULL THEN 'Quantity is missing'
        WHEN unit_price < 0 THEN 'Unit price is negative'
        WHEN order_date > CURRENT_DATE THEN 'Order date is in the future'
        WHEN store_id NOT IN (SELECT store_id FROM raw_stores) THEN 'Orphan store_id'
    END AS bad_reason
FROM raw_orders
WHERE 
    quantity IS NULL 
    OR unit_price < 0 
    OR order_date > CURRENT_DATE
    OR store_id NOT IN (SELECT store_id FROM raw_stores)
"""

con.execute(query_quarantine)


con.sql("SELECT order_id, store_id, quantity, unit_price, order_date, bad_reason FROM quarantine_orders").df()

,order_id,store_id,quantity,unit_price,order_date,bad_reason
0,4,1,NaN,200,2024-01-09,Quantity is missing
1,7,99,2.0,180,2024-01-16,Orphan store_id
2,10,3,2.0,-50,2024-01-23,Unit price is negative
3,15,2,NaN,150,2024-02-06,Quantity is missing
4,19,3,1.0,150,2099-12-31,Order date is in the future
5,22,99,1.0,120,2024-02-22,Orphan store_id
6,26,3,NaN,120,2024-03-04,Quantity is missing
7,28,2,1.0,-50,2024-03-10,Unit price is negative
8,31,2,NaN,120,2024-03-19,Quantity is missing


In [45]:
con.sql("show tables")

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ quarantine_orders │
│ raw_employees     │
│ raw_orders        │
│ raw_products      │
│ raw_stores        │
└───────────────────┘

In [46]:

con.sql("select * from raw_orders").df()


,order_id,store_id,product_id,employee_id,quantity,unit_price,order_date
0,1,1,101,1001,2.0,120,2024-01-03
1,2,2,102,1003,1.0,150,2024-01-05
2,3,3,103,1004,3.0,180,2024-01-07
3,4,1,104,1002,NaN,200,2024-01-09
4,5,2,105,1003,2.0,120,2024-01-11
5,6,3,101,1004,1.0,150,2024-01-14
6,7,99,102,1001,2.0,180,2024-01-16
7,8,1,103,1001,3.0,200,2024-01-18
8,9,2,104,1003,1.0,120,2024-01-21
9,10,3,105,1004,2.0,-50,2024-01-23


In [49]:
con.execute("""CREATE OR REPLACE TABLE clean_orders AS
SELECT * FROM raw_orders
WHERE 
    quantity IS NOT NULL 
    AND unit_price >= 0 
    AND order_date <= CURRENT_DATE
    AND store_id IN (SELECT store_id FROM raw_stores)
""")




con.sql("SELECT COUNT(*) as clean_rows FROM clean_orders").df()

,clean_rows
0,26


In [50]:
con.sql("select * from clean_orders")

┌──────────┬──────────┬────────────┬─────────────┬──────────┬────────────┬─────────────────────┐
│ order_id │ store_id │ product_id │ employee_id │ quantity │ unit_price │     order_date      │
│  int64   │  int64   │   int64    │    int64    │  double  │   int64    │      timestamp      │
├──────────┼──────────┼────────────┼─────────────┼──────────┼────────────┼─────────────────────┤
│        1 │        1 │        101 │        1001 │      2.0 │        120 │ 2024-01-03 00:00:00 │
│        2 │        2 │        102 │        1003 │      1.0 │        150 │ 2024-01-05 00:00:00 │
│        3 │        3 │        103 │        1004 │      3.0 │        180 │ 2024-01-07 00:00:00 │
│        5 │        2 │        105 │        1003 │      2.0 │        120 │ 2024-01-11 00:00:00 │
│        6 │        3 │        101 │        1004 │      1.0 │        150 │ 2024-01-14 00:00:00 │
│        8 │        1 │        103 │        1001 │      3.0 │        200 │ 2024-01-18 00:00:00 │
│        9 │        2 │       

In [51]:
query_summary = """
CREATE OR REPLACE TABLE pipeline_summary AS
SELECT 
    'raw_orders' AS table_name,
    (SELECT COUNT(*) FROM raw_orders) AS rows_in,
    (SELECT COUNT(*) FROM clean_orders) AS rows_passed,
    (SELECT COUNT(*) FROM quarantine_orders) AS rows_quarantined;
"""

con.execute(query_summary)


con.sql("SELECT * FROM pipeline_summary").df()

,table_name,rows_in,rows_passed,rows_quarantined
0,raw_orders,35,26,9
